# Avaliação de Desempenho: Poda Estruturada de Camadas (*Layer Pruning*) no AnyGraph

Este notebook avalia o impacto da **Poda Estruturada de Camadas (*Layer Pruning*)** baseada no **Block Influence (BI)** nos modelos pré-treinados do AnyGraph (`pretrain_link1` e `pretrain_link2`), comparando o desempenho do modelo original com as versões reduzidas.

### Configurações Avaliadas:
1. **Modelo Inteiro (Baseline):** 8 camadas ativas por expert (0% de poda — 16.87M parâmetros).
2. **Poda Tier 1 (Conservadora):** Remoção das Camadas **`[5, 6]`** em todos os experts (**-25% de parâmetros** — 12.65M parâmetros).
3. **Poda Tier 2 (Agressiva):** Remoção das Camadas **`[4, 5, 6, 7]`** em todos os experts (**-50% de parâmetros** — 8.43M parâmetros).

### Datasets Oficiais Utilizados:
Utiliza os **5 datasets oficiais de classificação de nós do AnyGraph** (os mesmos do benchmark de *Global Magnitude Pruning*):
- **`arxiv`** (169.383 nós — artigos científicos)
- **`cora`** (25.191 nós — citações acadêmicas)
- **`home`** (9.795 nós — co-compra de produtos domésticos)
- **`pubmed`** (19.720 nós — publicações biomédicas)
- **`tech`** (47.431 nós — co-compra de tecnologia)

In [1]:
import os
import sys
import copy
import torch as t
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path

# Identifica o diretorio OpenGraph de forma portavel
cwd = os.getcwd()
if os.path.basename(cwd) == "OpenGraph":
    PROJECT_ROOT = cwd
elif os.path.exists(os.path.join(cwd, "OpenGraph")):
    PROJECT_ROOT = os.path.join(cwd, "OpenGraph")
else:
    PROJECT_ROOT = os.path.abspath(".")

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
sys.argv = ["pruning_evaluation"]

from node_classification.params import args

# Usa GPU para velocidade máxima de avaliação
if t.cuda.is_available():
    args.gpu = "0"
    args.devices = ["cuda:0", "cuda:0"]
    print(f"Utilizando Dispositivo: {args.devices[0]} (GPU)")
else:
    args.gpu = "-1"
    args.devices = ["cpu", "cpu"]
    print("Utilizando Dispositivo: CPU")

args.tst_mode = "tst"
args.trn_mode = "train-all"
args.shot = 0
args.proj_method = "both"
args.gnn_layer = 3
args.latdim = 512

import node_classification.Utils.TimeLogger as logger
from node_classification.Utils.TimeLogger import log

from node_classification.data_handler import *
import node_classification.model
from node_classification.main import *

from Pruning_Methods.layer_pruner import *
sys.modules["model"] = node_classification.model
# Diretorios de saida
OUT_DIR = os.path.join(os.path.abspath(".."), "Block_Influence_results", "layer_pruning_evaluation")
os.makedirs(OUT_DIR, exist_ok=True)

# Datasets oficiais do benchmark
OFFICIAL_NODE_DATASETS = ["cora", "pubmed", "citeseer"]
print(f"Datasets Oficiais de Avaliação: {OFFICIAL_NODE_DATASETS}")
print(f"Diretório de Resultados: {OUT_DIR}")

Utilizando Dispositivo: cuda:0 (GPU)
Datasets Oficiais de Avaliação: ['cora', 'pubmed', 'citeseer']
Diretório de Resultados: c:\Mestrado\Graph_Pruning\Block_Influence_results\layer_pruning_evaluation


## 1. Pipeline Automatizado de Poda e Avaliação de Desempenho

In [3]:
def run_layer_pruning_experiment(
    model_checkpoint="pretrn_gen0",
    datasets=None,
    repeat_times=10
):
    """
    Executa o experimento de pruning baseado em Block Influence
    de forma isolada para cada dataset.

    Para cada dataset:
        1. Cria um MultiDataHandler novo
        2. Cria um Exp novo
        3. Avalia o Baseline
        4. Calcula Block Influence
        5. Tier 1: remove 1 camada com menor BI
        6. Tier 2: remove 2 camadas com menor BI
        7. Avalia cada configuração repeat_times vezes
    """

    if datasets is None:
        datasets = OFFICIAL_NODE_DATASETS

    results = []

    # Configuração do modelo
    args.load_model = model_checkpoint
    args.latdim = 1024

    # ============================================================
    # LOOP PRINCIPAL: UM EXP NOVO PARA CADA DATASET
    # ============================================================

    for dataset_name in datasets:

        print("\n" + "=" * 70)
        print(f"DATASET: {dataset_name}")
        print("=" * 70)

        # --------------------------------------------------------
        # Criar handler NOVO para este dataset
        # --------------------------------------------------------

        handler_data = MultiDataHandler(
            [dataset_name],
            [dataset_name]
        )

        # Criar Exp NOVO
        exp = Exp(handler_data)
        exp.prepare_model()

        # Único handler de teste deste dataset
        handler = exp.multi_handler.tst_handlers[0]

        # ========================================================
        # BASELINE
        # ========================================================

        print("\n" + "-" * 50)
        print("BASELINE")
        print("-" * 50)

        # Garantir que o modelo original seja carregado
        args.load_model = model_checkpoint
        exp.load_model()

        baseline_acc = []
        baseline_f1 = []

        for repeat in range(repeat_times):

            ret = exp.test_epoch(
                handler.tst_loader,
                handler
            )

            baseline_acc.append(ret["Acc"])
            baseline_f1.append(ret["F1"])

        baseline_acc_mean = np.mean(baseline_acc)
        baseline_acc_std = np.std(baseline_acc)

        baseline_f1_mean = np.mean(baseline_f1)
        baseline_f1_std = np.std(baseline_f1)

        print(
            f"Acc: {baseline_acc_mean:.4f} ± "
            f"{baseline_acc_std:.4f}"
        )

        print(
            f"F1:  {baseline_f1_mean:.4f} ± "
            f"{baseline_f1_std:.4f}"
        )

        # Salvar resultado do baseline
        results.append({
            "Configuração": "Baseline",
            "Camadas Removidas": "Nenhuma",
            "Camadas Restantes": "0,1,2,3",
            "Redução Parâmetros (%)": 0.0,
            "Dataset": dataset_name,
            "Acc_mean": baseline_acc_mean,
            "Acc_std": baseline_acc_std,
            "F1_mean": baseline_f1_mean,
            "F1_std": baseline_f1_std
        })

        # ========================================================
        # CALCULAR BLOCK INFLUENCE
        # ========================================================

        print("\n" + "-" * 50)
        print("BLOCK INFLUENCE")
        print("-" * 50)

        # Recarregar modelo original antes do cálculo do BI
        args.load_model = model_checkpoint
        exp.load_model()

        adj = handler.torch_adj
        initial_projector = handler.initial_projector

        if args.cache_adj == 0:
            adj = adj.to(args.devices[0])

        if args.cache_proj == 0:
            initial_projector = initial_projector.to(
                args.devices[0]
            )

        with t.no_grad():

            initial_embeds = initial_projector()

            input_embeds = exp.model.topoEncoder(
                adj,
                initial_embeds
            ).to(args.devices[1])

        # Criar pruner a partir do modelo original
        pruner = BlockInfluenceLayerPruner(
            exp.model
        )

        df_bi = pruner.compute_bi(
            input_embeds,
            angular=False
        )

        print("\nBlock Influence:")
        print(df_bi)

        # Ordenar da MENOR para a MAIOR influência
        ranking = (
            df_bi
            .sort_values("Block Influence")
            ["Layer"]
            .tolist()
        )

        print("\nRanking das camadas (menor BI → maior BI):")
        print(ranking)

        # ========================================================
        # TIER 1
        # ========================================================

        layers_t1 = ranking[:1]

        print("\n" + "-" * 50)
        print(f"TIER 1")
        print(f"Camada removida: {layers_t1}")
        print("-" * 50)

        # --------------------------------------------------------
        # IMPORTANTE:
        # Recarregar o modelo ORIGINAL.
        # Não reutilizar o modelo usado no BI.
        # --------------------------------------------------------

        args.load_model = model_checkpoint
        exp.load_model()

        pruner_t1 = BlockInfluenceLayerPruner(
            exp.model
        )

        pruned_model_t1 = (
            pruner_t1.prune_specific_layers(
                layers_t1
            )
        )

        exp.model = pruned_model_t1

        # Relatório de sparsity
        report_t1 = pruner_t1.sparsity_report()

        print("Sparsity report Tier 1:")
        print(report_t1)

        # Avaliação
        t1_acc = []
        t1_f1 = []

        for repeat in range(repeat_times):

            ret = exp.test_epoch(
                handler.tst_loader,
                handler
            )

            t1_acc.append(ret["Acc"])
            t1_f1.append(ret["F1"])

        t1_acc_mean = np.mean(t1_acc)
        t1_acc_std = np.std(t1_acc)

        t1_f1_mean = np.mean(t1_f1)
        t1_f1_std = np.std(t1_f1)

        print(
            f"Acc: {t1_acc_mean:.4f} ± "
            f"{t1_acc_std:.4f}"
        )

        print(
            f"F1:  {t1_f1_mean:.4f} ± "
            f"{t1_f1_std:.4f}"
        )

        # Salvar resultado
        results.append({
            "Configuração": "Tier 1",
            "Camadas Removidas": str(layers_t1),
            "Camadas Restantes": str(
                [
                    i
                    for i in range(4)
                    if i not in layers_t1
                ]
            ),
            "Redução Parâmetros (%)": report_t1,
            "Dataset": dataset_name,
            "Acc_mean": t1_acc_mean,
            "Acc_std": t1_acc_std,
            "F1_mean": t1_f1_mean,
            "F1_std": t1_f1_std
        })

        # ========================================================
        # TIER 2
        # ========================================================

        layers_t2 = ranking[:2]

        print("\n" + "-" * 50)
        print(f"TIER 2")
        print(f"Camadas removidas: {layers_t2}")
        print("-" * 50)

        # --------------------------------------------------------
        # Recarregar novamente o modelo ORIGINAL
        # --------------------------------------------------------

        args.load_model = model_checkpoint
        exp.load_model()

        pruner_t2 = BlockInfluenceLayerPruner(
            exp.model
        )

        pruned_model_t2 = (
            pruner_t2.prune_specific_layers(
                layers_t2
            )
        )

        exp.model = pruned_model_t2

        # Relatório de sparsity
        report_t2 = pruner_t2.sparsity_report()

        print("Sparsity report Tier 2:")
        print(report_t2)

        # Avaliação
        t2_acc = []
        t2_f1 = []

        for repeat in range(repeat_times):

            ret = exp.test_epoch(
                handler.tst_loader,
                handler
            )

            t2_acc.append(ret["Acc"])
            t2_f1.append(ret["F1"])

        t2_acc_mean = np.mean(t2_acc)
        t2_acc_std = np.std(t2_acc)

        t2_f1_mean = np.mean(t2_f1)
        t2_f1_std = np.std(t2_f1)

        print(
            f"Acc: {t2_acc_mean:.4f} ± "
            f"{t2_acc_std:.4f}"
        )

        print(
            f"F1:  {t2_f1_mean:.4f} ± "
            f"{t2_f1_std:.4f}"
        )

        # Salvar resultado
        results.append({
            "Configuração": "Tier 2",
            "Camadas Removidas": str(layers_t2),
            "Camadas Restantes": str(
                [
                    i
                    for i in range(4)
                    if i not in layers_t2
                ]
            ),
            "Redução Parâmetros (%)": report_t2,
            "Dataset": dataset_name,
            "Acc_mean": t2_acc_mean,
            "Acc_std": t2_acc_std,
            "F1_mean": t2_f1_mean,
            "F1_std": t2_f1_std
        })

        # ========================================================
        # RESUMO DO DATASET
        # ========================================================

        print("\n" + "=" * 70)
        print(f"RESUMO - {dataset_name}")
        print("=" * 70)

        print(
            f"Baseline : "
            f"Acc={baseline_acc_mean:.4f} ± {baseline_acc_std:.4f} | "
            f"F1={baseline_f1_mean:.4f} ± {baseline_f1_std:.4f}"
        )

        print(
            f"Tier 1   : "
            f"Acc={t1_acc_mean:.4f} ± {t1_acc_std:.4f} | "
            f"F1={t1_f1_mean:.4f} ± {t1_f1_std:.4f} | "
            f"Remove={layers_t1}"
        )

        print(
            f"Tier 2   : "
            f"Acc={t2_acc_mean:.4f} ± {t2_acc_std:.4f} | "
            f"F1={t2_f1_mean:.4f} ± {t2_f1_std:.4f} | "
            f"Remove={layers_t2}"
        )

    # ============================================================
    # DATAFRAME FINAL
    # ============================================================

    results_df = pd.DataFrame(results)

    print("\n" + "=" * 70)
    print("RESULTADOS FINAIS")
    print("=" * 70)

    display(results_df)


    # ============================================================
    # SALVAR CSV
    # ============================================================

    output_dir = Path(OUT_DIR)

    output_path = output_dir / (
        f"layer_pruning_{model_checkpoint}.csv"
    )

    results_df.to_csv(
        output_path,
        index=False
    )

    print(
        f"\nResultados salvos em:\n{output_path}"
    )

    return results_df

## 2. Avaliação de Poda de Camadas no Modelo 1 (`pretrn_gen0`)

In [4]:
datasets = [
    "cora",
    "pubmed",
    "citeseer"
]

results_df1 = run_layer_pruning_experiment(
    model_checkpoint="pretrn_gen0",
    datasets=datasets,
    repeat_times=10
)
print("=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 1 ===")
display(results_df1)


DATASET: cora
Dataset: cora, Node num: 2715, Edge num: 11836


c:\Mestrado\Graph_Pruning\OpenGraph\node_classification\data_handler.py:115: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  asym_adj = t.sparse_coo_tensor(idxs, vals, shape, check_invariants=False)


Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0

--------------------------------------------------
BASELINE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-11 15:48:09.887852: Model Loaded
Acc: 0.4560 ± 0.0094471768: Steps 3/3: hit = 464, tot = 1000          
F1:  0.4412 ± 0.0108

--------------------------------------------------
BLOCK INFLUENCE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-11 15:48:10.659341: Model Loaded

Block Influence:
   Layer  Block Influence
0      0         0.045974
1      1         0.016020
2      2         0.029051
3      3         0.047010

Ranking das camadas (menor BI → maior BI):
[1, 2, 

c:\Mestrado\Graph_Pruning\OpenGraph\node_classification\data_handler.py:62: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])


Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0

--------------------------------------------------
BASELINE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-11 15:48:22.071948: Model Loaded
Acc: 0.5604 ± 0.0075438984: Steps 3/3: hit = 547, tot = 1000          
F1:  0.5364 ± 0.0068

--------------------------------------------------
BLOCK INFLUENCE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen0.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen0.his
2026-09-11 15:48:22.625944: Model Loaded

Block Influence:
   Layer  Block Influence
0      0         0.067512
1      1         0.012682
2      2         0.032852
3      3         0.048894

Ranking das camadas (menor BI → maior BI):
[1, 2, 

,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Baseline,Nenhuma,"0,1,2,3",0.0,cora,0.4560,0.009402,0.441225,0.010835
1,Tier 1,[1],"[0, 2, 3]","{'Total Params Original': 25190400, 'Total Par...",cora,0.5140,0.015120,0.502092,0.015320
2,Tier 2,"[1, 2]","[0, 3]","{'Total Params Original': 25190400, 'Total Par...",cora,0.5366,0.012706,0.528851,0.010760
3,Baseline,Nenhuma,"0,1,2,3",0.0,pubmed,0.4517,0.035426,0.425442,0.037641
4,Tier 1,[1],"[0, 2, 3]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.4788,0.024653,0.434269,0.030777
5,Tier 2,"[1, 2]","[0, 3]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.4717,0.026526,0.428298,0.037990
6,Baseline,Nenhuma,"0,1,2,3",0.0,citeseer,0.5604,0.007526,0.536407,0.006792
7,Tier 1,[1],"[0, 2, 3]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5678,0.007467,0.544902,0.008004
8,Tier 2,"[1, 2]","[0, 3]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5625,0.009394,0.541044,0.008347



Resultados salvos em:
c:\Mestrado\Graph_Pruning\Block_Influence_results\layer_pruning_evaluation\layer_pruning_pretrn_gen0.csv
=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 1 ===


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Baseline,Nenhuma,"0,1,2,3",0.0,cora,0.4560,0.009402,0.441225,0.010835
1,Tier 1,[1],"[0, 2, 3]","{'Total Params Original': 25190400, 'Total Par...",cora,0.5140,0.015120,0.502092,0.015320
2,Tier 2,"[1, 2]","[0, 3]","{'Total Params Original': 25190400, 'Total Par...",cora,0.5366,0.012706,0.528851,0.010760
3,Baseline,Nenhuma,"0,1,2,3",0.0,pubmed,0.4517,0.035426,0.425442,0.037641
4,Tier 1,[1],"[0, 2, 3]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.4788,0.024653,0.434269,0.030777
5,Tier 2,"[1, 2]","[0, 3]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.4717,0.026526,0.428298,0.037990
6,Baseline,Nenhuma,"0,1,2,3",0.0,citeseer,0.5604,0.007526,0.536407,0.006792
7,Tier 1,[1],"[0, 2, 3]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5678,0.007467,0.544902,0.008004
8,Tier 2,"[1, 2]","[0, 3]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5625,0.009394,0.541044,0.008347


## 3. Avaliação de Poda de Camadas no Modelo 2 (`pretrn_gen1`)

In [5]:
datasets = [
    "cora",
    "pubmed",
    "citeseer"
]

results_df2 = run_layer_pruning_experiment(
    model_checkpoint="pretrn_gen1",
    datasets=datasets,
    repeat_times=10
)
print("=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 2 ===")
display(results_df2)
display(results_df2)


DATASET: cora
Dataset: cora, Node num: 2715, Edge num: 11836
Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0

--------------------------------------------------
BASELINE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-11 15:48:46.488934: Model Loaded
Acc: 0.7441 ± 0.0042808042: Steps 3/3: hit = 747, tot = 1000          
F1:  0.7366 ± 0.0048

--------------------------------------------------
BLOCK INFLUENCE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-11 15:48:46.870648: Model Loaded

Block Influence:
   Layer  Block Influence
0      0         0.040757
1      1         0.015557
2      2         0.008707
3      3       

c:\Mestrado\Graph_Pruning\OpenGraph\node_classification\data_handler.py:62: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])


Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0

--------------------------------------------------
BASELINE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-11 15:48:53.079608: Model Loaded
Acc: 0.5964 ± 0.0091477331: Steps 3/3: hit = 609, tot = 1000          
F1:  0.5686 ± 0.0095

--------------------------------------------------
BLOCK INFLUENCE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-11 15:48:53.544567: Model Loaded

Block Influence:
   Layer  Block Influence
0      0         0.092556
1      1         0.029293
2      2         0.016172
3      3         0.008418

Ranking das camadas (menor BI → maior BI):
[3, 2, 

,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Baseline,Nenhuma,"0,1,2,3",0.0,cora,0.7441,0.004182,0.736578,0.004832
1,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7449,0.006007,0.737747,0.006055
2,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7408,0.004874,0.734316,0.005293
3,Baseline,Nenhuma,"0,1,2,3",0.0,pubmed,0.6938,0.019052,0.667941,0.022960
4,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.7000,0.017135,0.676369,0.016609
5,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.6864,0.028510,0.662999,0.026694
6,Baseline,Nenhuma,"0,1,2,3",0.0,citeseer,0.5964,0.009124,0.568629,0.009479
7,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5995,0.007117,0.571446,0.005378
8,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5976,0.005696,0.566530,0.006161



Resultados salvos em:
c:\Mestrado\Graph_Pruning\Block_Influence_results\layer_pruning_evaluation\layer_pruning_pretrn_gen1.csv
=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 2 ===


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Baseline,Nenhuma,"0,1,2,3",0.0,cora,0.7441,0.004182,0.736578,0.004832
1,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7449,0.006007,0.737747,0.006055
2,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7408,0.004874,0.734316,0.005293
3,Baseline,Nenhuma,"0,1,2,3",0.0,pubmed,0.6938,0.019052,0.667941,0.022960
4,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.7000,0.017135,0.676369,0.016609
5,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.6864,0.028510,0.662999,0.026694
6,Baseline,Nenhuma,"0,1,2,3",0.0,citeseer,0.5964,0.009124,0.568629,0.009479
7,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5995,0.007117,0.571446,0.005378
8,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5976,0.005696,0.566530,0.006161


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Baseline,Nenhuma,"0,1,2,3",0.0,cora,0.7441,0.004182,0.736578,0.004832
1,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7449,0.006007,0.737747,0.006055
2,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7408,0.004874,0.734316,0.005293
3,Baseline,Nenhuma,"0,1,2,3",0.0,pubmed,0.6938,0.019052,0.667941,0.022960
4,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.7000,0.017135,0.676369,0.016609
5,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.6864,0.028510,0.662999,0.026694
6,Baseline,Nenhuma,"0,1,2,3",0.0,citeseer,0.5964,0.009124,0.568629,0.009479
7,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5995,0.007117,0.571446,0.005378
8,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5976,0.005696,0.566530,0.006161


## 4. Avaliação de Poda de Camadas no Modelo 3 (`pretrn_gen2`)

In [6]:
datasets = [
    "cora",
    "pubmed",
    "citeseer"
]

results_df3 = run_layer_pruning_experiment(
    model_checkpoint="pretrn_gen2",
    datasets=datasets,
    repeat_times=10
)
print("=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 3 ===")
display(results_df3)
display(results_df3)


DATASET: cora
Dataset: cora, Node num: 2715, Edge num: 11836
Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0

--------------------------------------------------
BASELINE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen2.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen2.his
2026-09-11 15:49:18.717944: Model Loaded
Acc: 0.7568 ± 0.0049058929: Steps 3/3: hit = 750, tot = 1000          
F1:  0.7482 ± 0.0049

--------------------------------------------------
BLOCK INFLUENCE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen2.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen2.his
2026-09-11 15:49:19.119053: Model Loaded

Block Influence:
   Layer  Block Influence
0      0         0.054508
1      1         0.024902
2      2         0.014241
3      3       

c:\Mestrado\Graph_Pruning\OpenGraph\node_classification\data_handler.py:62: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])


Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0

--------------------------------------------------
BASELINE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen2.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen2.his
2026-09-11 15:49:25.323693: Model Loaded
Acc: 0.5627 ± 0.0154703077: Steps 3/3: hit = 554, tot = 1000          
F1:  0.5296 ± 0.0155

--------------------------------------------------
BLOCK INFLUENCE
--------------------------------------------------
Loading model from: c:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen2.mod
Loading history from: c:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen2.his
2026-09-11 15:49:25.769348: Model Loaded

Block Influence:
   Layer  Block Influence
0      0         0.109796
1      1         0.049691
2      2         0.034222
3      3         0.015207

Ranking das camadas (menor BI → maior BI):
[3, 2, 

,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Baseline,Nenhuma,"0,1,2,3",0.0,cora,0.7568,0.004874,0.748170,0.004862
1,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7595,0.003528,0.752541,0.003233
2,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7573,0.004713,0.748122,0.003675
3,Baseline,Nenhuma,"0,1,2,3",0.0,pubmed,0.6877,0.055160,0.663536,0.047537
4,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.6905,0.035322,0.662745,0.033448
5,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.6990,0.022751,0.666272,0.021518
6,Baseline,Nenhuma,"0,1,2,3",0.0,citeseer,0.5627,0.015428,0.529561,0.015488
7,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5858,0.013511,0.549650,0.013570
8,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5976,0.012249,0.560539,0.012318



Resultados salvos em:
c:\Mestrado\Graph_Pruning\Block_Influence_results\layer_pruning_evaluation\layer_pruning_pretrn_gen2.csv
=== TABELA CONSOLIDADA DE DESEMPENHO - MODELO 3 ===


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Baseline,Nenhuma,"0,1,2,3",0.0,cora,0.7568,0.004874,0.748170,0.004862
1,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7595,0.003528,0.752541,0.003233
2,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7573,0.004713,0.748122,0.003675
3,Baseline,Nenhuma,"0,1,2,3",0.0,pubmed,0.6877,0.055160,0.663536,0.047537
4,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.6905,0.035322,0.662745,0.033448
5,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.6990,0.022751,0.666272,0.021518
6,Baseline,Nenhuma,"0,1,2,3",0.0,citeseer,0.5627,0.015428,0.529561,0.015488
7,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5858,0.013511,0.549650,0.013570
8,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5976,0.012249,0.560539,0.012318


,Configuração,Camadas Removidas,Camadas Restantes,Redução Parâmetros (%),Dataset,Acc_mean,Acc_std,F1_mean,F1_std
0,Baseline,Nenhuma,"0,1,2,3",0.0,cora,0.7568,0.004874,0.748170,0.004862
1,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7595,0.003528,0.752541,0.003233
2,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",cora,0.7573,0.004713,0.748122,0.003675
3,Baseline,Nenhuma,"0,1,2,3",0.0,pubmed,0.6877,0.055160,0.663536,0.047537
4,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.6905,0.035322,0.662745,0.033448
5,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",pubmed,0.6990,0.022751,0.666272,0.021518
6,Baseline,Nenhuma,"0,1,2,3",0.0,citeseer,0.5627,0.015428,0.529561,0.015488
7,Tier 1,[3],"[0, 1, 2]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5858,0.013511,0.549650,0.013570
8,Tier 2,"[3, 2]","[0, 1]","{'Total Params Original': 25190400, 'Total Par...",citeseer,0.5976,0.012249,0.560539,0.012318
